# Kapitel 5: Pre-Training

## Notwendige Bibliotheken importieren

In [21]:
# --- stdlib ---
import ast
import os
import sys
from datetime import datetime

# --- third-party ---
import torch
import tiktoken
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output
from tqdm.auto import tqdm

# --- eigene utils (gpt.py im utils-Ordner) ---
sys.path.append(os.path.join(os.getcwd(), "utils"))
from gpt import (
    GPTModel,
    generate,
    create_dataloader_v1,
    load_weights_into_gpt,   
)

## Den Datensatz aus der Cloud importieren

In [22]:
# Reduzierter Datensatz mit 100k Zeilen zum Testen
cloud_url = "https://syncandshare.lrz.de/dl/fiHE8nDPcb4nww3VCn4QmN/reduced_dataset_100k.csv"
# Die folgende Zeile einkommentieren, um den vollständigen Datensatz zu verwenden
# cloud_url = "https://syncandshare.lrz.de/dl/fiHE8nDPcb4nww3VCn4QmN/full_dataset.csv"
try:
    df = pd.read_csv(cloud_url)
    df = df.head(25000)   # NUR TEST — für echten/Cloud-Lauf wieder entfernen
except Exception as e:
    print(f'{e}')

if __name__ == '__main__':

    try:
        print("Lade Datensatz aus der Cloud...")
        #df = pd.read_csv(cloud_url)
        print("Datensatz erfolgreich geladen!\n")
        print("Info:")
        print(df.info())
        print("")
        print("Anfang (Head) des Datensatzes:")
        print(df.head())
        
    except Exception as e:
        print(f"Ein Fehler ist beim Laden des Datensatzes aufgetreten: {e}")

Lade Datensatz aus der Cloud...
Datensatz erfolgreich geladen!

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   25000 non-null  int64 
 1   title        25000 non-null  object
 2   ingredients  25000 non-null  object
 3   directions   25000 non-null  object
 4   link         25000 non-null  object
 5   source       25000 non-null  object
 6   NER          25000 non-null  object
dtypes: int64(1), object(6)
memory usage: 1.3+ MB
None

Anfang (Head) des Datensatzes:
   Unnamed: 0                  title  \
0           0    No-Bake Nut Cookies   
1           1  Jewell Ball'S Chicken   
2           2            Creamy Corn   
3           3          Chicken Funny   
4           4   Reeses Cups(Candy)     

                                         ingredients  \
0  ["1 c. firmly packed brown sugar", "1/2 c. eva...   
1  ["1 small 

## Trainingskorpus mit <|endoftext|> Tokens generieren.
- Wir arbeiten hier mit (semi-)strukturierten Rezeptdaten. 
- Jedes Rezept wir stellt eigenes Dokument dar, deshalb wird nach jedem Rezept ein ein <|endoftext|> Token eingefügt

In [23]:
def to_list(cell):
    if isinstance(cell, list): return cell
    if pd.isna(cell): return []
    try: return ast.literal_eval(cell)
    except (ValueError, SyntaxError): return [str(cell)]

def format_recipe(row):
    title = str(row["title"]).strip()
    ingr  = "\n".join(s.strip() for s in to_list(row["ingredients"]))
    direc = "\n".join(s.strip() for s in to_list(row["directions"]))
    return f"Recipe: {title}\nIngredients:\n{ingr}\nDirections:\n{direc}"

recipes = [format_recipe(r) for _, r in df.iterrows()]
corpus = "".join(r + "<|endoftext|>" for r in recipes)

with open("datasets/pre-training/corpus.txt", "w", encoding="utf-8") as f:
    f.write(corpus)
print(len(recipes), "Rezepte;", len(corpus), "Zeichen")


25000 Rezepte; 11762991 Zeichen


#### Train/Validierungs-Split
- Der erstellte Korpus wird in Training (90 %) und Validierung (10 %) aufgeteilt
- Gesplittet wird entlang des Separators <|endoftext|> (auf Rezept-Ebene), nicht per Zeichen-Index. So bleibt jedes Rezept vollständig, kein Rezept oder Token wird mitten durchgeschnitten und jedes Rezept endet sauber mit dem <|endoftext|>-Token.

In [24]:
sys.path.append(os.path.join(os.getcwd(), "utils"))
from gpt import GPTDatasetV1, create_dataloader_v1

SEP = "<|endoftext|>"
with open("datasets/pre-training/corpus.txt", "r", encoding="utf-8") as f:
    corpus = f.read()

# auf REZEPT-Ebene splitten (Separator = Grenze), nicht per Zeichen-Index
recipes = [r for r in corpus.split(SEP) if r.strip()]
split_idx = int(0.9 * len(recipes))
train_data = "".join(r + SEP for r in recipes[:split_idx])
val_data   = "".join(r + SEP for r in recipes[split_idx:])

print(f"Rezepte gesamt: {len(recipes)}")
print(f"Training:    {len(recipes[:split_idx])} Rezepte, {len(train_data)} Zeichen")
print(f"Validierung: {len(recipes[split_idx:])} Rezepte, {len(val_data)} Zeichen")

Rezepte gesamt: 25000
Training:    22500 Rezepte, 10586954 Zeichen
Validierung: 2500 Rezepte, 1176037 Zeichen


# DataLoader

Der DataLoader aus Lab 2 wird für Training und Validierung konfiguriert.
Jeder Batch enthält 2 Sequenzen mit je 256 Tokens, entsprechend der
konfigurierten context_length. Da stride = context_length gesetzt ist,
werden die Sequenzen ohne Überlappung aneinandergereiht. Der
Trainingsdatensatz wird mit shuffle=True zufällig gemischt, um eine
Überanpassung an die Reihenfolge der Daten zu vermeiden, während der
Validierungsdatensatz sequenziell verarbeitet wird. drop_last=True stellt
sicher, dass unvollständige Batches beim Training verworfen werden.

In [25]:
GPT_CONFIG = {
    "vocab_size": 50257,     # MUSS zum gpt2-Tokenizer passen (sonst KeyError beim Decoden)
    "context_length": 512,    
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False,
}


In [26]:

torch.manual_seed(123)

train_loader = create_dataloader_v1(
    train_data,
    batch_size=4,
    max_length=GPT_CONFIG["context_length"],
    stride=GPT_CONFIG["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = create_dataloader_v1(
    val_data,
    batch_size=4,
    max_length=GPT_CONFIG["context_length"],
    stride=GPT_CONFIG["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)
    break

print("Validation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)
    break

Train loader:
torch.Size([4, 512]) torch.Size([4, 512])
Validation loader:
torch.Size([4, 512]) torch.Size([4, 512])


# Verlustfunktion

Implementierung der Verlustfunktionen zur Bewertung der Modellleistung.

calc_loss_batch berechnet den Cross-Entropy Loss für einen einzelnen 
Batch. Der Cross-Entropy Loss misst die Abweichung zwischen den 
vorhergesagten Token-Wahrscheinlichkeiten und den tatsächlichen 
Ziel-Tokens. Je niedriger der Loss desto besser die Vorhersagen des Modells.

calc_loss_loader berechnet den durchschnittlichen Loss über mehrere 
Batches eines DataLoaders und ermöglicht so eine Gesamtbewertung der 
Modellleistung auf dem Trainings- oder Validierungsdatensatz.

In [27]:
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch, target_batch = input_batch.to(device), target_batch.to(device)
    logits = model(input_batch)
    return torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())


def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i >= num_batches:
            break
        total_loss += calc_loss_batch(input_batch, target_batch, model, device).item()
    return total_loss / num_batches

# Device Konfiguration und initialer Loss

Das Modell wird auf dem verfügbaren Gerät initialisiert. Dabei wird
bevorzugt eine CUDA GPU verwendet, gefolgt von Apple MPS für M-Series
Prozessoren und CPU als Fallback.

Der initiale Loss wird vor dem Training berechnet, um einen Referenzwert
zu erhalten. Bei zufällig initialisierten Gewichten verteilt das Modell
die Wahrscheinlichkeit annähernd gleichmäßig über alle Tokens. Der
erwartete Cross-Entropy-Loss entspricht daher ln(vocab_size) =
ln(50257) ≈ 10.82. Der gemessene Wert von ~10.97 liegt in dieser
Größenordnung und bestätigt, dass das untrainierte Modell noch keine
Struktur gelernt hat.

In [28]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using {device} device.")

model = GPTModel(GPT_CONFIG)
model.to(device)

torch.manual_seed(123)

with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device, num_batches=5)
    val_loss   = calc_loss_loader(val_loader,   model, device, num_batches=5)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Using mps device.
Training loss: 10.977009201049805
Validation loss: 10.969866180419922


# Hilfsfunktionen zur Textkonvertierung

text_to_token_ids kodiert einen Text mittels tiktoken in Token IDs 
und fügt eine Batch-Dimension hinzu, da das Modell einen 2D Tensor 
als Eingabe erwartet.

token_ids_to_text dekodiert Token IDs zurück in lesbaren Text 
indem die Batch-Dimension entfernt und die IDs mittels tiktoken 
in Zeichenketten umgewandelt werden.

In [29]:
def text_to_token_ids(text, tokenizer):
    return torch.tensor(tokenizer.encode(text)).unsqueeze(0)

def token_ids_to_text(token_ids, tokenizer):
    return tokenizer.decode(token_ids.squeeze(0).tolist())

# Evaluierungs- und Sampling Funktionen

evaluate_model berechnet den Loss auf Trainings- und Validierungsdaten 
während des Trainings. Das Modell wird dabei in den Evaluierungsmodus 
versetzt um Dropout zu deaktivieren, und anschließend wieder in den 
Trainingsmodus zurückgesetzt.

generate_and_print_sample generiert nach jeder Trainingsepoche einen 
Beispieltext um den Lernfortschritt des Modells qualitativ zu beurteilen. 
Der generierte Text zeigt wie gut das Modell bereits sinnvolle 
Rezeptstrukturen erlernt hat.

In [30]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss


def generate_and_print_sample(model, tokenizer, device, start_context, max_new_tokens=50, temperature=0.8, top_k=25):
    model.eval()
    with torch.no_grad():
        token_ids = generate(
            model=model,
            idx=text_to_token_ids(start_context, tokenizer).to(device),
            max_new_tokens=max_new_tokens,
            context_size=model.pos_emb.weight.shape[0],
            temperature=temperature,   
            top_k=top_k,
        )
    print(token_ids_to_text(token_ids, tokenizer).replace("\n", " "))
    model.train()


# Live Visualisierung des Trainingsverlaufs

Der Live Plot aktualisiert sich nach jedem Evaluierungsschritt während 
des Trainings und zeigt den Verlauf von Training Loss und Validation Loss. 
clear_output löscht dabei die vorherige Ausgabe um eine animierte 
Darstellung zu erzeugen. So kann der Lernfortschritt des Modells in 
Echtzeit beobachtet werden und ein frühzeitiges Overfitting erkannt werden.


In [31]:

from IPython.display import clear_output
import matplotlib.pyplot as plt

def plot_losses_live(train_losses, val_losses, tokens_seen):
    clear_output(wait=True)
    clear_output(wait=True)
    fig, ax1 = plt.subplots(figsize=(5, 3))
    ax1.plot(train_losses, label="Training loss")
    ax1.plot(val_losses, linestyle="-.", label="Validation loss")
    ax1.set_xlabel("Steps")
    ax1.set_ylabel("Loss")
    ax1.legend()
    plt.tight_layout()
    plt.show()
    


# Training

Implementierung der Trainingsschleife mittels train_model_simple. 
In jeder Iteration wird der Loss berechnet, die Gradienten durch 
loss.backward() propagiert und die Gewichte durch den AdamW Optimizer 
aktualisiert. AdamW wird gegenüber dem klassischen SGD bevorzugt da 
er adaptive Lernraten verwendet und Weight Decay zur Regularisierung 
integriert.

Alle eval_freq Schritte wird der aktuelle Loss auf Trainings- und 
Validierungsdaten berechnet und der Live Plot aktualisiert. Nach jeder 
Epoche wird ein Beispieltext generiert um den qualitativen Lernfortschritt 
zu beurteilen.

## Overfitting Analyse

Nach dem Training wird überprüft ob der Validation Loss am Ende höher 
ist als zu Beginn. Ein steigender Validation Loss bei sinkendem Training 
Loss deutet auf Overfitting hin, das Modell hat die Trainingsdaten 
auswendig gelernt anstatt allgemeine Sprachmuster zu erkennen. Bei einem 
kleinen Datensatz wie hier ist Overfitting ein erwartetes Verhalten.

In [ ]:
def train_model_simple(model, train_loader, val_loader, optimizer, device, num_epochs,
                       eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1
    best_val_loss = float("inf")          # Early Stopping
    best_state = None

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in tqdm(train_loader, desc=f"Epoche {epoch+1}/{num_epochs}"):
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)   # Grad-Clipping
            optimizer.step()
            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                tqdm.write(f"Ep {epoch+1} (Step {global_step}): Train {train_loss:.3f} | Val {val_loss:.3f}")

                if val_loss < best_val_loss:                                    # Early Stopping: bestes Modell merken
                    best_val_loss = val_loss
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        generate_and_print_sample(model, tokenizer, device, start_context)

    if best_state is not None:                                                  # bestes statt letztes Modell behalten
        model.load_state_dict(best_state)
        print(f"\nBestes Modell wiederhergestellt (Val-Loss {best_val_loss:.3f}).")
    return train_losses, val_losses, track_tokens_seen


tokenizer = tiktoken.get_encoding("gpt2")

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

num_epochs = 7
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=200, eval_iter=50,
    start_context="Recipe:", tokenizer=tokenizer
)

# Plot einmal am Ende (ohne clear_output -> Log + Plot bleiben sichtbar)
plt.figure(figsize=(6, 4))
plt.plot(train_losses, label="Train loss")
plt.plot(val_losses, linestyle="-.", label="Val loss")
plt.xlabel("Eval-Schritt"); plt.ylabel("Loss"); plt.legend(); plt.tight_layout(); plt.show()

# Overfitting-Report
if val_losses[-1] > min(val_losses):
    print(f"Val-Loss-Minimum: {min(val_losses):.3f} | am Ende: {val_losses[-1]:.3f} → Overfitting nach dem Tiefpunkt")
else:
    print("Val-Loss noch fallend — evtl. mehr Daten/Epochen möglich")

Epoche 1/7:   0%|          | 0/1479 [00:00<?, ?it/s]

Ep 1 (Step 0): Train 8.858 | Val 8.862
Ep 1 (Step 200): Train 2.899 | Val 2.920
Ep 1 (Step 400): Train 2.564 | Val 2.616
Ep 1 (Step 600): Train 2.407 | Val 2.445
Ep 1 (Step 800): Train 2.240 | Val 2.314
Ep 1 (Step 1000): Train 2.138 | Val 2.233
Ep 1 (Step 1200): Train 2.082 | Val 2.162
Ep 1 (Step 1400): Train 1.979 | Val 2.104
Recipe: Butterscotchinoed Ingredients: 2 c. white Karo 1 tsp. salt 1/2 tsp. pepper 2 c. sugar 1/2 c. chopped pecans 1/2 c. chopped


Epoche 2/7:   0%|          | 0/1479 [00:00<?, ?it/s]

Ep 2 (Step 1600): Train 1.932 | Val 2.054
Ep 2 (Step 1800): Train 1.879 | Val 2.011
Ep 2 (Step 2000): Train 1.823 | Val 1.985
Ep 2 (Step 2200): Train 1.812 | Val 1.962
Ep 2 (Step 2400): Train 1.767 | Val 1.926
Ep 2 (Step 2600): Train 1.719 | Val 1.903
Ep 2 (Step 2800): Train 1.695 | Val 1.882
Recipe: Chicken Casserole Ingredients: 2 1/2 c. flour 4 c. milk 3/4 c. chicken broth 1 c. water 1/4 tsp. Worcestershire sauce 1/4 tsp.


Epoche 3/7:   0%|          | 0/1479 [00:00<?, ?it/s]

Ep 3 (Step 3000): Train 1.694 | Val 1.855
Ep 3 (Step 3200): Train 1.600 | Val 1.847
Ep 3 (Step 3400): Train 1.581 | Val 1.841
Ep 3 (Step 3600): Train 1.592 | Val 1.827
Ep 3 (Step 3800): Train 1.517 | Val 1.814
Ep 3 (Step 4000): Train 1.511 | Val 1.803
Ep 3 (Step 4200): Train 1.507 | Val 1.792
Ep 3 (Step 4400): Train 1.472 | Val 1.775
Recipe: Tbsp. butter Ingredients: 1/2 c. flour 1/2 c. sugar 1 egg, beaten 3 Tbsp. water 1/2 tsp. baking powder 1/2 tsp. salt 1/


Epoche 4/7:   0%|          | 0/1479 [00:00<?, ?it/s]

Ep 4 (Step 4600): Train 1.445 | Val 1.781
Ep 4 (Step 4800): Train 1.427 | Val 1.793
Ep 4 (Step 5000): Train 1.450 | Val 1.777
Ep 4 (Step 5200): Train 1.403 | Val 1.772
Ep 4 (Step 5400): Train 1.378 | Val 1.760
Ep 4 (Step 5600): Train 1.369 | Val 1.752
Ep 4 (Step 5800): Train 1.305 | Val 1.741
Recipe: Corn Casserole Ingredients: 1 can cream-style corn 1 c. milk 1/4 c. yellow corn meal 1 egg 1 1/2 c. biscuit mix 1/3 c. margarine 


Epoche 5/7:   0%|          | 0/1479 [00:00<?, ?it/s]

Ep 5 (Step 6000): Train 1.291 | Val 1.746
Ep 5 (Step 6200): Train 1.269 | Val 1.765


# Erweiterte Textgenerierung

Implementierung der generate Funktion als Erweiterung von 
generate_text_simple mit zwei zusätzlichen Dekodierungsstrategien.

## Top-k Sampling

Top-k Sampling beschränkt die Auswahl des nächsten Tokens auf die 
k wahrscheinlichsten Kandidaten. Alle anderen Tokens werden auf 
-inf gesetzt und nach der Softmax-Funktion zu 0. Dies verhindert 
dass unwahrscheinliche Tokens ausgewählt werden.

## Temperature Scaling

Temperature skaliert die Logits vor der Softmax-Funktion. Eine niedrige 
Temperature macht die Verteilung schärfer und das Modell wählt 
konservativer das wahrscheinlichste Token. Eine hohe Temperature 
verteilt die Wahrscheinlichkeiten gleichmäßiger und erzeugt 
kreativeren aber weniger kohärenten Text. Bei temperature=0.0 
wird Greedy Decoding verwendet wie in generate_text_simple.

In [ ]:
def generate(model, idx, max_new_tokens, context_size, temperature=0.0, top_k=None, eos_id=None):

    # For-loop is the same as before: Get logits, and only focus on last time step
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]

        # New: Filter logits with top_k sampling
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]
            logits = torch.where(logits < min_val, torch.tensor(float("-inf")).to(logits.device), logits)

        # New: Apply temperature scaling
        if temperature > 0.0:
            logits = logits / temperature
            logits = logits - logits.max(dim=-1, keepdim=True).values
            probs = torch.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)

        # Otherwise same as before: get idx of the vocab entry with the highest logits value
        else:
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        if idx_next == eos_id:
            break

        idx = torch.cat((idx, idx_next), dim=1)

    return idx

# Vergleich verschiedener Temperaturen

Der Vergleich zeigt den Einfluss der Temperature auf den generierten Text. 
Bei temperature=0.0 wird stets das wahrscheinlichste Token gewählt, 
was zu deterministischem aber repetitivem Text führt. Mit steigender 
Temperature wird die Ausgabe zunehmend variabler und kreativer, 
jedoch auf Kosten der sprachlichen Kohärenz. temperature=1.4 erzeugt 
den kreativsten Text, kann jedoch auch zu weniger sinnvollen 
Wortfolgen führen.

In [ ]:
temperatures = [0.0, 0.5, 1.4]

for temp in temperatures:
    torch.manual_seed(123)
    token_ids = generate(
        model=model,
        idx=text_to_token_ids("Recipe:", tokenizer).to(device),
        max_new_tokens=20,
        context_size=GPT_CONFIG["context_length"],
        top_k=25,
        temperature=temp
    )
    print(temp)
    print(token_ids_to_text(token_ids, tokenizer))

# Generierung verschiedener Rezepte

Das trainierte Modell wird mit verschiedenen Rezeptanfängen getestet 
um die Generalisierungsfähigkeit zu evaluieren. Jeder Kontext gibt 
einen Rezepttitel und den Beginn der Zutatenliste vor, woraufhin 
das Modell die Fortsetzung generiert.

Da das Modell auf Rezeptdaten trainiert wurde, sollte es grundlegende 
Rezeptstrukturen wie Zutatenmengen und Zubereitungsschritte erkennen. 
Die Qualität der generierten Rezepte hängt dabei direkt von der 
Trainingsdatenmenge und der Anzahl der Trainingsepochen ab. Bei einem 
kleinen Datensatz und wenigen Epochen sind die Ausgaben noch 
inkohärent, zeigen jedoch bereits grundlegende Rezeptmuster.

In [ ]:
contexts = [
    "Recipe: Chocolate Cake\nIngredients:",
    "Recipe: Pizza\nIngredients:",
    "Recipe: Soup\nIngredients:",
]

for context in contexts:
    torch.manual_seed(123)
    token_ids = generate(
        model=model,
        idx=text_to_token_ids(context, tokenizer).to(device),
        max_new_tokens=100,
        context_size=GPT_CONFIG["context_length"],
        top_k=25,
        temperature=1.4
    )
    print(f"\nStart: {context}")
    print(f"Generated: {token_ids_to_text(token_ids, tokenizer)}")

# Textgenerierung mit optimierten Parametern

Abschließende Textgenerierung mit den optimierten Dekodierungsparametern 
top_k=25 und temperature=1.4. Diese Kombination aus Top-k Sampling 
und Temperature Scaling erzeugt einen ausgewogenen Text der sowohl 
kohärent als auch kreativ ist. Der generierte Text zeigt den 
aktuellen Lernstand des Modells nach dem Training auf dem Rezeptdatensatz.

In [ ]:
torch.manual_seed(123)

token_ids = generate(
    model=model,
    idx=text_to_token_ids("Recipe:", tokenizer).to(device),
    max_new_tokens=15,
    context_size=GPT_CONFIG["context_length"],
    top_k=25,
    temperature=1.4
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
best_val = min(val_losses)
RUN_NAME = f"model_pretrained_r{len(recipes)}_ctx{GPT_CONFIG['context_length']}_e{num_epochs}_val{best_val:.2f}_{ts}"
MODEL_PATH = f"datasets/pre-training/{RUN_NAME}.pth"
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": GPT_CONFIG,
    "train_losses": train_losses,
    "val_losses": val_losses,
    "num_epochs": num_epochs,
}, MODEL_PATH)
print(f"Modell gespeichert: {MODEL_PATH}")


## OpenAI-GPT-2-Gewichte laden & vergleichen

#### 1) Gewichte herunterladen & ins Modell laden
download_and_load_gpt2 lädt den offiziellen GPT-2-Checkpoint (TensorFlow-Format)
von OpenAI und gibt settings (Hyperparameter) + params (Gewichts-Arrays) zurück.
load_weights_into_gpt kopiert diese Arrays Position für Position in unsere
eigene GPTModel-Instanz (Embeddings → jeder Transformer-Block → finale Norm → Output-Head).

Die Config muss exakt zu GPT-2-small passen, sonst Shape-Mismatch:
-  qkv_bias=True -> GPT-2 nutzt Bias in der Attention; fehlen diese Parameter, schlägt das Zuweisen fehl.
- context_length=1024 -> GPT-2s Positions-Embedding ist [1024, 768]; mit 256 → Shape-Fehler.
- drop_rate=0.0 -> wir evaluieren nur, kein Dropout nötig.

In [ ]:
from gpt_download import download_and_load_gpt2

# Lädt den GPT-2-small-Checkpoint (124M) — gleiche Architektur wie unser Modell
settings, params = download_and_load_gpt2(model_size="124M", models_dir="gpt2")

# Config exakt nach GPT-2-small (Abweichungen zum Trainingsmodell markiert)
GPT2_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,   # GPT-2: 1024 (nicht 256!) — sonst Shape-Mismatch bei pos_emb
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.0,         # nur Eval
    "qkv_bias": True,         # GPT-2 hat Attention-Bias -> MUSS True sein
}

gpt2 = GPTModel(GPT2_CONFIG)
load_weights_into_gpt(gpt2, params)   # OpenAI-Gewichte in unsere Architektur kopieren
gpt2.to(device)
gpt2.eval()
print("GPT-2 (124M) geladen.")

#### 2) Train-/Validierungs-Loss auf unseren Rezeptdaten
Wir nutzen  exakt dieselbe calc_loss_loader-Funktion und dieselben
train_loader/val_loader wie für unser eigenes Modell, nur das Modell ist
ausgetauscht. Dadurch sind die Loss-Werte direkt vergleichbar.
(Unsere Loader liefern 256-Token-Sequenzen; GPT-2 verträgt bis 1024 → passt.)

**Erwartung:** GPT-2 hat unsere Rezepte nie gesehen, ist aber auf riesigen
Textmengen trainiert → sein Loss sollte deutlich niedriger liegen als der
unseres From-Scratch-Modells. 

In [ ]:
torch.manual_seed(123)
with torch.no_grad():
    gpt2_train_loss = calc_loss_loader(train_loader, gpt2, device, num_batches=20)
    gpt2_val_loss   = calc_loss_loader(val_loader,   gpt2, device, num_batches=20)

print("=== Loss-Vergleich auf unseren Rezeptdaten ===")
print(f"From-Scratch ({len(recipes)}   -> Train: {train_losses[-1]:.3f} | Val: {val_losses[-1]:.3f}")
print(f"GPT-2 OpenAI (124M, Web-Text) -> Train: {gpt2_train_loss:.3f} | Val: {gpt2_val_loss:.3f}")

#### 3) Textgenerierung
Wir rufen dieselbe generate()-Funktion (Top-k + Temperature) auf wie oben,
nur mit gpt2 als Modell. eos_id setzen wir auf das `<|endoftext|>`-Token,
damit die Generierung am Dokumentende stoppen kann.

In [ ]:
eot_id = tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})[0]

for context in ["Recipe: Chocolate Cake\nIngredients:", "Recipe: Pizza\nIngredients:"]:
    torch.manual_seed(123)
    token_ids = generate(
        model=gpt2,
        idx=text_to_token_ids(context, tokenizer).to(device),
        max_new_tokens=100,
        context_size=GPT2_CONFIG["context_length"],
        top_k=25,
        temperature=1.0,
        eos_id=eot_id,
    )
    print(f"\nStart: {context}")
    print(f"Generated: {token_ids_to_text(token_ids, tokenizer)}")